# FFN 与 Block 精简版

> 本文是 [ch05.ipynb](./ch05.ipynb) 的浓缩版。只保留 SwiGLU 核心代码、shape 流动和 Block 结构。

## SwiGLU 核心代码

```python
class FeedForward(nn.Module):
    def __init__(self, config):
        self.gate_proj = nn.Linear(768, 2432, bias=False)
        self.down_proj = nn.Linear(2432, 768, bias=False)
        self.up_proj   = nn.Linear(768, 2432, bias=False)
        self.act_fn    = F.silu  # silu(x) = x * sigmoid(x)

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
```

## Shape 流动

```
(b, T, 768) ─gate_proj→ (b, T, 2432) ─silu→ ──┐
                                               × ─down_proj→ (b, T, 768)
(b, T, 768) ─up_proj──→ (b, T, 2432) ─────────┘
```

参数量:$3 \times 768 \times 2432 = 5{,}603{,}328$ / 层

## Block 结构

```
h ─┬─ input_ln ─→ Attention ─→ + ─┬─ post_attn_ln ─→ FFN ─→ + ─→ h_out
   └──── residual 1 ──────────────┘└──── residual 2 ───────────┘
```

两个 Pre-Norm 残差子层,残差流始终保持 `(b, T, 768)`。

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

class FeedForward(nn.Module):
    def __init__(self, d=768, d_ff=None):
        super().__init__()
        d_ff = d_ff or math.ceil(d * math.pi / 64) * 64
        self.gate_proj = nn.Linear(d, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d, bias=False)
        self.up_proj   = nn.Linear(d, d_ff, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

ffn = FeedForward()
x = torch.randn(1, 5, 768)
print(f"in:  {x.shape}")
print(f"out: {ffn(x).shape}")
print(f"params: {sum(p.numel() for p in ffn.parameters()):,}")

## Dense vs MoE

| 指标 | Dense | MoE (4 experts) |
|---|---|---|
| FFN 参数/层 | 5.6M | 22.4M |
| 每 token 激活 | 5.6M | 5.6M(相同!) |
| 全模型 | 64M | 198M |

MoE = 3 倍容量,相同计算成本。`aux_loss` 保证负载均衡。